# 🧠 NLP Model Comparison Dashboard — Research Notebook

> **Dataset:** Stanford Sentiment Treebank v2 (SST-2) via HuggingFace `datasets`  
> **Goal:** Compare NLTK, spaCy, and Transformer pipelines for binary sentiment classification  
> **Author:** NLP Comparison Dashboard Project

---

## 📑 Table of Contents
1. [Setup & Imports](#1-setup)
2. [Dataset Loading & EDA](#2-eda)
3. [NLTK Pipeline](#3-nltk)
4. [spaCy Pipeline](#4-spacy)
5. [Transformer Pipeline](#5-transformer)
6. [Model Comparison](#6-comparison)
7. [Live Inference Demo](#7-inference)
8. [Save Models](#8-save)

---
## 1. 🔧 Setup & Imports <a id='1-setup'></a>

In [ ]:
# ── Install dependencies (uncomment if not already installed) ──────────────
# !pip install nltk spacy transformers torch scikit-learn datasets plotly joblib
# !python -m spacy download en_core_web_sm

In [ ]:
import os, re, time, warnings
warnings.filterwarnings('ignore')

# ── Data & ML ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import joblib
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_curve, auc, ConfusionMatrixDisplay
)

# ── Visualisation ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    'figure.facecolor': '#0d0f1a',
    'axes.facecolor':   '#111827',
    'axes.edgecolor':   '#374151',
    'axes.labelcolor':  '#e2e8f0',
    'xtick.color':      '#94a3b8',
    'ytick.color':      '#94a3b8',
    'text.color':       '#e2e8f0',
    'grid.color':       '#1f2937',
    'figure.titlesize': 16,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ── NLP ────────────────────────────────────────────────────────────────────
import nltk
nltk.download('punkt',     quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import spacy

MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

print('✅ All imports successful')
print(f'   NumPy  : {np.__version__}')
print(f'   Pandas : {pd.__version__}')
print(f'   spaCy  : {spacy.__version__}')
print(f'   sklearn: {__import__("sklearn").__version__}')

---
## 2. 📊 Dataset Loading & EDA <a id='2-eda'></a>

In [ ]:
# ── Load SST-2 ─────────────────────────────────────────────────────────────
print('📦 Loading SST-2 dataset …')
dataset_train = load_dataset('sst2', split='train')
dataset_val   = load_dataset('sst2', split='validation')

df_train = pd.DataFrame({'text': dataset_train['sentence'], 'label': dataset_train['label']})
df_val   = pd.DataFrame({'text': dataset_val['sentence'],   'label': dataset_val['label']})

print(f'Train set : {len(df_train):,} samples')
print(f'Val   set : {len(df_val):,}  samples')
df_train.head(8)

In [ ]:
# ── Basic stats ────────────────────────────────────────────────────────────
df_train['word_count'] = df_train['text'].apply(lambda x: len(x.split()))
df_train['char_count'] = df_train['text'].apply(len)
df_train['label_name'] = df_train['label'].map({0: 'Negative', 1: 'Positive'})

print('=== Overall Stats ===')
print(df_train[['word_count', 'char_count']].describe().round(2))
print()
print('=== Label Distribution ===')
print(df_train['label_name'].value_counts())

In [ ]:
# ── EDA Charts ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('SST-2 Dataset — Exploratory Data Analysis', fontsize=15, fontweight='bold', color='#c7d2fe')

COLORS = {'Negative': '#f87171', 'Positive': '#4ade80'}

# 1. class balance
counts = df_train['label_name'].value_counts()
axes[0].bar(counts.index, counts.values,
            color=[COLORS[l] for l in counts.index],
            edgecolor='#1f2937', linewidth=1.2)
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, (lbl, val) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, val + 100, f'{val:,}', ha='center', fontsize=10, color='#94a3b8')

# 2. word count distribution
for lbl, grp in df_train.groupby('label_name'):
    axes[1].hist(grp['word_count'], bins=30, alpha=0.7, label=lbl, color=COLORS[lbl], edgecolor='#1f2937')
axes[1].set_title('Word Count Distribution')
axes[1].set_xlabel('Words per sentence')
axes[1].set_ylabel('Frequency')
axes[1].legend()

# 3. char count distribution
for lbl, grp in df_train.groupby('label_name'):
    axes[2].hist(grp['char_count'], bins=30, alpha=0.7, label=lbl, color=COLORS[lbl], edgecolor='#1f2937')
axes[2].set_title('Character Count Distribution')
axes[2].set_xlabel('Characters per sentence')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Sample sentences per class ─────────────────────────────────────────────
print('🟢 POSITIVE samples:')
for t in df_train[df_train.label == 1]['text'].sample(4, random_state=1).values:
    print('  ✅', t)

print()
print('🔴 NEGATIVE samples:')
for t in df_train[df_train.label == 0]['text'].sample(4, random_state=1).values:
    print('  ❌', t)

In [ ]:
# ── Train / test split ─────────────────────────────────────────────────────
X = df_train['text'].tolist()
y = df_train['label'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train : {len(X_train):,}   |  Pos: {sum(y_train):,}  Neg: {len(y_train)-sum(y_train):,}')
print(f'Test  : {len(X_test):,}    |  Pos: {sum(y_test):,}   Neg: {len(y_test)-sum(y_test):,}')

---
## 3. 🔵 NLTK Pipeline <a id='3-nltk'></a>

**Flow:** Raw text → NLTK tokenisation → stopword removal → TF-IDF → Logistic Regression

In [ ]:
# ── Preprocessing ──────────────────────────────────────────────────────────
STOPWORDS_EN = set(stopwords.words('english'))

def preprocess_nltk(text: str) -> str:
    text   = text.lower()
    text   = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS_EN and len(t) > 1]
    return ' '.join(tokens)

# demo
sample = 'This movie is not bad at all — I genuinely enjoyed it!'
print('Original :', sample)
print('Processed:', preprocess_nltk(sample))

In [ ]:
# ── Preprocessing impact (token count before vs after) ─────────────────────
sample_texts = df_train['text'].sample(5, random_state=7).tolist()
comparison = []
for txt in sample_texts:
    proc = preprocess_nltk(txt)
    comparison.append({
        'Original': txt[:70] + '…' if len(txt) > 70 else txt,
        'Raw tokens': len(txt.split()),
        'After NLTK': len(proc.split()),
        'Reduction': f"{100*(1 - len(proc.split())/max(1,len(txt.split()))):.0f}%"
    })
pd.DataFrame(comparison)

In [ ]:
# ── Apply preprocessing ────────────────────────────────────────────────────
print('Preprocessing train set …')
t0 = time.time()
X_train_nltk = [preprocess_nltk(t) for t in X_train]
X_test_nltk  = [preprocess_nltk(t) for t in X_test]
print(f'  Done in {time.time()-t0:.1f}s')

# ── TF-IDF ─────────────────────────────────────────────────────────────────
vec_nltk = TfidfVectorizer(max_features=20_000, ngram_range=(1, 2))
X_train_vec = vec_nltk.fit_transform(X_train_nltk)
X_test_vec  = vec_nltk.transform(X_test_nltk)

print(f'  Vocabulary size : {len(vec_nltk.vocabulary_):,}')
print(f'  Feature matrix  : {X_train_vec.shape}')

In [ ]:
# ── Train Logistic Regression ───────────────────────────────────────────────
print('Training Logistic Regression (NLTK) …')
t0 = time.time()
model_nltk = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs')
model_nltk.fit(X_train_vec, y_train)
print(f'  Done in {time.time()-t0:.1f}s')

# ── Evaluate ───────────────────────────────────────────────────────────────
preds_nltk = model_nltk.predict(X_test_vec)
proba_nltk = model_nltk.predict_proba(X_test_vec)[:, 1]
acc_nltk   = accuracy_score(y_test, preds_nltk)

print(f'\n📊 NLTK Accuracy: {acc_nltk:.4f}')
print(classification_report(y_test, preds_nltk, target_names=['Negative', 'Positive']))

In [ ]:
# ── Top TF-IDF features ────────────────────────────────────────────────────
feature_names = vec_nltk.get_feature_names_out()
coef          = model_nltk.coef_[0]
top_pos_idx   = coef.argsort()[-15:][::-1]
top_neg_idx   = coef.argsort()[:15]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('NLTK · Top TF-IDF Feature Weights (Logistic Regression)', color='#c7d2fe')

# positive
axes[0].barh(
    feature_names[top_pos_idx][::-1],
    coef[top_pos_idx][::-1],
    color='#4ade80', edgecolor='#1f2937'
)
axes[0].set_title('Top POSITIVE features')
axes[0].set_xlabel('Coefficient weight')

# negative
axes[1].barh(
    feature_names[top_neg_idx],
    coef[top_neg_idx],
    color='#f87171', edgecolor='#1f2937'
)
axes[1].set_title('Top NEGATIVE features')
axes[1].set_xlabel('Coefficient weight')

plt.tight_layout()
plt.show()

In [ ]:
# ── NLTK Confusion Matrix ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, preds_nltk)
disp = ConfusionMatrixDisplay(cm, display_labels=['Negative', 'Positive'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('NLTK — Confusion Matrix')
plt.tight_layout()
plt.show()

---
## 4. 🟢 spaCy Pipeline <a id='4-spacy'></a>

**Flow:** Raw text → spaCy lemmatisation → stopword/punct removal → TF-IDF → Logistic Regression

In [ ]:
# ── Load spaCy model ───────────────────────────────────────────────────────
try:
    nlp_spacy = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
except OSError:
    import subprocess
    subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'])
    nlp_spacy = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

print(f'spaCy model: {nlp_spacy.meta["name"]} v{nlp_spacy.meta["version"]}')
print(f'Pipeline   : {nlp_spacy.pipe_names}')

In [ ]:
# ── spaCy preprocessing ────────────────────────────────────────────────────
def preprocess_spacy(text: str) -> str:
    doc    = nlp_spacy(text.lower())
    tokens = [
        tok.lemma_ for tok in doc
        if not tok.is_stop and not tok.is_punct and len(tok.text) > 1
    ]
    return ' '.join(tokens)

sample = 'This movie is not bad at all — I genuinely enjoyed it!'
print('Original   :', sample)
print('NLTK proc  :', preprocess_nltk(sample))
print('spaCy proc :', preprocess_spacy(sample))

In [ ]:
# ── preprocessing comparison side-by-side ──────────────────────────────────
rows = []
for txt in df_train['text'].sample(6, random_state=42).tolist():
    rows.append({
        'Original'  : txt[:60] + '…' if len(txt) > 60 else txt,
        'NLTK'      : preprocess_nltk(txt),
        'spaCy'     : preprocess_spacy(txt),
    })
pd.set_option('display.max_colwidth', 80)
pd.DataFrame(rows)

In [ ]:
# ── Apply & vectorise ──────────────────────────────────────────────────────
print('Preprocessing with spaCy …')
t0 = time.time()
X_train_spacy = [preprocess_spacy(t) for t in X_train]
X_test_spacy  = [preprocess_spacy(t) for t in X_test]
print(f'  Done in {time.time()-t0:.1f}s')

vec_spacy = TfidfVectorizer(max_features=20_000, ngram_range=(1, 2))
X_train_vec_sp = vec_spacy.fit_transform(X_train_spacy)
X_test_vec_sp  = vec_spacy.transform(X_test_spacy)

print(f'  Vocabulary size : {len(vec_spacy.vocabulary_):,}')
print(f'  Feature matrix  : {X_train_vec_sp.shape}')

In [ ]:
# ── Train & evaluate ───────────────────────────────────────────────────────
print('Training Logistic Regression (spaCy) …')
t0 = time.time()
model_spacy = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs')
model_spacy.fit(X_train_vec_sp, y_train)
print(f'  Done in {time.time()-t0:.1f}s')

preds_spacy = model_spacy.predict(X_test_vec_sp)
proba_spacy = model_spacy.predict_proba(X_test_vec_sp)[:, 1]
acc_spacy   = accuracy_score(y_test, preds_spacy)

print(f'\n📊 spaCy Accuracy: {acc_spacy:.4f}')
print(classification_report(y_test, preds_spacy, target_names=['Negative', 'Positive']))

In [ ]:
# ── Top spaCy features ─────────────────────────────────────────────────────
feature_names_sp = vec_spacy.get_feature_names_out()
coef_sp          = model_spacy.coef_[0]
top_pos_sp       = coef_sp.argsort()[-15:][::-1]
top_neg_sp       = coef_sp.argsort()[:15]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('spaCy · Top TF-IDF Feature Weights (Logistic Regression)', color='#c7d2fe')

axes[0].barh(feature_names_sp[top_pos_sp][::-1], coef_sp[top_pos_sp][::-1],
             color='#4ade80', edgecolor='#1f2937')
axes[0].set_title('Top POSITIVE features')

axes[1].barh(feature_names_sp[top_neg_sp], coef_sp[top_neg_sp],
             color='#f87171', edgecolor='#1f2937')
axes[1].set_title('Top NEGATIVE features')

plt.tight_layout()
plt.show()

In [ ]:
# ── spaCy Confusion Matrix ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
cm_sp = confusion_matrix(y_test, preds_spacy)
ConfusionMatrixDisplay(cm_sp, display_labels=['Negative', 'Positive']).plot(ax=ax, colorbar=False, cmap='Greens')
ax.set_title('spaCy — Confusion Matrix')
plt.tight_layout()
plt.show()

---
## 5. 🔴 Transformer Pipeline (DistilBERT) <a id='5-transformer'></a>

**Flow:** Raw text → BERT WordPiece tokenisation (internal) → DistilBERT fine-tuned on SST-2

In [ ]:
from transformers import pipeline as hf_pipeline

print('Loading DistilBERT pipeline … (first run downloads ~270 MB)')
t0 = time.time()
classifier = hf_pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english',
    device=-1,          # CPU; set to 0 for GPU
    truncation=True,
    max_length=512,
)
print(f'  Loaded in {time.time()-t0:.1f}s')

In [ ]:
# ── Single inference demo ──────────────────────────────────────────────────
demos = [
    'This movie is not bad at all — I genuinely enjoyed it!',
    'Absolutely terrible. Would not recommend to anyone.',
    'The acting was decent but the plot was quite boring.',
    'One of the best films I\'ve seen this year.',
]

print(f'{"Sentence":<55} {"Model":12} {"Prob":>6}')
print('-' * 75)
for txt in demos:
    r = classifier(txt)[0]
    print(f"{txt[:54]:<55} {r['label']:12} {r['score']:.4f}")

In [ ]:
# ── Evaluate on test subset (first 800 for speed) ─────────────────────────
print('Running Transformer on test subset (800 samples) …')
TEST_N      = 800
X_sub       = X_test[:TEST_N]
y_sub       = y_test[:TEST_N]

t0          = time.time()
raw_results = classifier(X_sub, batch_size=32, truncation=True, max_length=512)
elapsed_bert = time.time() - t0

preds_bert  = [1 if r['label'] == 'POSITIVE' else 0 for r in raw_results]
proba_bert  = [r['score'] if r['label'] == 'POSITIVE' else 1 - r['score'] for r in raw_results]
acc_bert    = accuracy_score(y_sub, preds_bert)

print(f'  {TEST_N} samples in {elapsed_bert:.1f}s  ({elapsed_bert/TEST_N*1000:.1f} ms/sample)')
print(f'\n📊 Transformer Accuracy (subset): {acc_bert:.4f}')
print(classification_report(y_sub, preds_bert, target_names=['Negative', 'Positive']))

In [ ]:
# ── DistilBERT Confusion Matrix ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
cm_bert = confusion_matrix(y_sub, preds_bert)
ConfusionMatrixDisplay(cm_bert, display_labels=['Negative', 'Positive']).plot(ax=ax, colorbar=False, cmap='RdPu')
ax.set_title('DistilBERT — Confusion Matrix (800-sample subset)')
plt.tight_layout()
plt.show()

---
## 6. 📈 Model Comparison <a id='6-comparison'></a>

In [ ]:
# ── ROC Curves ─────────────────────────────────────────────────────────────
# align test sets (use same 800-sample subset for fairness)
X_sub_nltk  = X_test_vec[:TEST_N]
X_sub_sp    = X_test_vec_sp[:TEST_N]

proba_nltk_sub  = model_nltk.predict_proba(X_sub_nltk)[:, 1]
proba_spacy_sub = model_spacy.predict_proba(X_sub_sp)[:, 1]

fig, ax = plt.subplots(figsize=(7, 6))
fig.patch.set_facecolor('#0d0f1a')

for name, proba, color in [
    ('NLTK',        proba_nltk_sub,  '#60a5fa'),
    ('spaCy',       proba_spacy_sub, '#34d399'),
    ('Transformer', proba_bert,      '#f472b6'),
]:
    fpr, tpr, _ = roc_curve(y_sub, proba)
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'gray', linestyle='--', lw=1, label='Random baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison (800-sample subset)', color='#c7d2fe')
ax.legend(loc='lower right', framealpha=0.2)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()

In [ ]:
# ── Accuracy bar chart ─────────────────────────────────────────────────────
# compute NLTK and spaCy accuracy on same subset
preds_nltk_sub  = model_nltk.predict(X_sub_nltk)
preds_spacy_sub = model_spacy.predict(X_sub_sp)
acc_nltk_sub    = accuracy_score(y_sub, preds_nltk_sub)
acc_spacy_sub   = accuracy_score(y_sub, preds_spacy_sub)

models  = ['🔵 NLTK', '🟢 spaCy', '🔴 DistilBERT']
accs    = [acc_nltk_sub, acc_spacy_sub, acc_bert]
colors  = ['#60a5fa', '#34d399', '#f472b6']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(models, accs, color=colors, edgecolor='#1f2937', width=0.5)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.2%}', ha='center', va='bottom', fontweight='bold')
ax.set_ylim(0.7, 1.0)
ax.set_ylabel('Accuracy')
ax.set_title('Model Accuracy Comparison (800-sample subset)', color='#c7d2fe')
ax.axhline(0.5, color='gray', linestyle='--', lw=1, label='Random baseline')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Speed comparison ───────────────────────────────────────────────────────
BENCH_N   = 100
bench_txt = X_test[:BENCH_N]

# NLTK
t0 = time.perf_counter()
for txt in bench_txt:
    proc = preprocess_nltk(txt)
    model_nltk.predict(vec_nltk.transform([proc]))
ms_nltk = (time.perf_counter() - t0) / BENCH_N * 1000

# spaCy
t0 = time.perf_counter()
for txt in bench_txt:
    proc = preprocess_spacy(txt)
    model_spacy.predict(vec_spacy.transform([proc]))
ms_spacy = (time.perf_counter() - t0) / BENCH_N * 1000

# Transformer
t0 = time.perf_counter()
for txt in bench_txt:
    classifier(txt, truncation=True, max_length=512)
ms_bert = (time.perf_counter() - t0) / BENCH_N * 1000

print(f'Average inference time per sample ({BENCH_N} samples):')
print(f'  NLTK        : {ms_nltk:.2f} ms')
print(f'  spaCy       : {ms_spacy:.2f} ms')
print(f'  Transformer : {ms_bert:.2f} ms')

fig, ax = plt.subplots(figsize=(7, 4))
times   = [ms_nltk, ms_spacy, ms_bert]
bars    = ax.bar(models, times, color=colors, edgecolor='#1f2937', width=0.5)
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{t:.1f} ms', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Time per sample (ms)')
ax.set_title('Inference Speed Comparison', color='#c7d2fe')
plt.tight_layout()
plt.show()

In [ ]:
# ── Plotly interactive comparison dashboard ────────────────────────────────
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Accuracy', 'Speed (ms/sample)'),
    horizontal_spacing=0.14
)

model_labels = ['NLTK', 'spaCy', 'DistilBERT']

fig.add_trace(go.Bar(
    x=model_labels, y=[acc_nltk_sub, acc_spacy_sub, acc_bert],
    marker_color=['#60a5fa', '#34d399', '#f472b6'],
    text=[f'{v:.1%}' for v in [acc_nltk_sub, acc_spacy_sub, acc_bert]],
    textposition='outside', name='Accuracy'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=model_labels, y=[ms_nltk, ms_spacy, ms_bert],
    marker_color=['#60a5fa', '#34d399', '#f472b6'],
    text=[f'{v:.1f} ms' for v in [ms_nltk, ms_spacy, ms_bert]],
    textposition='outside', name='Speed'
), row=1, col=2)

fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='rgba(13,15,26,1)',
    plot_bgcolor='rgba(17,24,39,1)',
    title='NLP Pipeline Comparison — Accuracy vs Speed',
    title_font=dict(size=16, color='#c7d2fe'),
    showlegend=False,
    font=dict(family='Inter', color='#e2e8f0'),
    height=420,
)
fig.update_yaxes(range=[0.7, 1.05], tickformat='.0%', row=1, col=1)
fig.show()

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────
from sklearn.metrics import f1_score, precision_score, recall_score

summary = pd.DataFrame([
    {
        'Model'     : '🔵 NLTK',
        'Accuracy'  : f'{acc_nltk_sub:.4f}',
        'Precision' : f'{precision_score(y_sub, preds_nltk_sub):.4f}',
        'Recall'    : f'{recall_score(y_sub, preds_nltk_sub):.4f}',
        'F1'        : f'{f1_score(y_sub, preds_nltk_sub):.4f}',
        'Speed'     : f'{ms_nltk:.1f} ms',
    },
    {
        'Model'     : '🟢 spaCy',
        'Accuracy'  : f'{acc_spacy_sub:.4f}',
        'Precision' : f'{precision_score(y_sub, preds_spacy_sub):.4f}',
        'Recall'    : f'{recall_score(y_sub, preds_spacy_sub):.4f}',
        'F1'        : f'{f1_score(y_sub, preds_spacy_sub):.4f}',
        'Speed'     : f'{ms_spacy:.1f} ms',
    },
    {
        'Model'     : '🔴 DistilBERT',
        'Accuracy'  : f'{acc_bert:.4f}',
        'Precision' : f'{precision_score(y_sub, preds_bert):.4f}',
        'Recall'    : f'{recall_score(y_sub, preds_bert):.4f}',
        'F1'        : f'{f1_score(y_sub, preds_bert):.4f}',
        'Speed'     : f'{ms_bert:.1f} ms',
    },
])

print('\n📋 Final Model Comparison Summary')
print('='*65)
summary

In [ ]:
# ── Error analysis — cases where models disagree ───────────────────────────
disagreements = []
for i in range(min(TEST_N, len(X_sub_nltk))):
    txt        = X_test[i]
    true       = y_sub[i]
    p_nltk     = model_nltk.predict(vec_nltk.transform([preprocess_nltk(txt)]))[0]
    p_spacy    = model_spacy.predict(vec_spacy.transform([preprocess_spacy(txt)]))[0]
    p_bert     = preds_bert[i]
    if not (p_nltk == p_spacy == p_bert):
        disagreements.append({
            'Text'    : txt[:70] + '…' if len(txt) > 70 else txt,
            'True'    : '✅ Pos' if true == 1 else '❌ Neg',
            'NLTK'    : '✅' if p_nltk == 1 else '❌',
            'spaCy'   : '✅' if p_spacy == 1 else '❌',
            'BERT'    : '✅' if p_bert == 1 else '❌',
        })

print(f'Models disagreed on {len(disagreements)} / {TEST_N} samples ({len(disagreements)/TEST_N:.1%})')
pd.DataFrame(disagreements).head(10)

---
## 7. 🔍 Live Inference Demo <a id='7-inference'></a>

In [ ]:
def predict_all(text: str) -> pd.DataFrame:
    """Run all 3 pipelines on a single text and return a comparison DataFrame."""
    rows = []

    # NLTK
    t0   = time.perf_counter()
    proc = preprocess_nltk(text)
    X    = vec_nltk.transform([proc])
    pred = model_nltk.predict(X)[0]
    conf = model_nltk.predict_proba(X)[0].max()
    rows.append({'Model': '🔵 NLTK', 'Prediction': 'Positive ✅' if pred==1 else 'Negative ❌',
                 'Confidence': f'{conf:.2%}', 'Time': f'{(time.perf_counter()-t0)*1000:.1f} ms'})

    # spaCy
    t0   = time.perf_counter()
    proc = preprocess_spacy(text)
    X    = vec_spacy.transform([proc])
    pred = model_spacy.predict(X)[0]
    conf = model_spacy.predict_proba(X)[0].max()
    rows.append({'Model': '🟢 spaCy', 'Prediction': 'Positive ✅' if pred==1 else 'Negative ❌',
                 'Confidence': f'{conf:.2%}', 'Time': f'{(time.perf_counter()-t0)*1000:.1f} ms'})

    # Transformer
    t0  = time.perf_counter()
    r   = classifier(text, truncation=True, max_length=512)[0]
    lbl = 'Positive ✅' if r['label']=='POSITIVE' else 'Negative ❌'
    rows.append({'Model': '🔴 DistilBERT', 'Prediction': lbl,
                 'Confidence': f"{r['score']:.2%}", 'Time': f'{(time.perf_counter()-t0)*1000:.1f} ms'})

    return pd.DataFrame(rows)


test_sentences = [
    'This movie is not bad at all — I genuinely enjoyed it!',
    'Absolutely terrible. Would not recommend to anyone.',
    'The acting was decent but the plot was quite boring.',
    'One of the best films I\'ve seen this year.',
    'It had its moments, but overall it left me cold.',
]

for sent in test_sentences:
    print(f'\n⬛ Sentence: "{sent}"')
    display(predict_all(sent))
    print()

---
## 8. 💾 Save Models <a id='8-save'></a>

In [ ]:
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

joblib.dump(model_nltk,   f'{MODELS_DIR}/nltk_model.pkl')
joblib.dump(vec_nltk,     f'{MODELS_DIR}/nltk_vectorizer.pkl')
joblib.dump(model_spacy,  f'{MODELS_DIR}/spacy_model.pkl')
joblib.dump(vec_spacy,    f'{MODELS_DIR}/spacy_vectorizer.pkl')

print('✅ Models saved:')
for f in os.listdir(MODELS_DIR):
    path = os.path.join(MODELS_DIR, f)
    size = os.path.getsize(path) / 1024
    print(f'   {f:<35} {size:.1f} KB')

In [ ]:
# ── Final summary card ─────────────────────────────────────────────────────
print('\n' + '='*60)
print('  📋  NLP PIPELINE COMPARISON — FINAL RESULTS')
print('='*60)
print(f'  Dataset    : SST-2 ({len(X_train)+len(X_test):,} train · {TEST_N} eval subset)')
print(f'  NLTK       : {acc_nltk_sub:.2%}  accuracy  |  {ms_nltk:.1f} ms/sample')
print(f'  spaCy      : {acc_spacy_sub:.2%}  accuracy  |  {ms_spacy:.1f} ms/sample')
print(f'  DistilBERT : {acc_bert:.2%}  accuracy  |  {ms_bert:.1f} ms/sample')
print()
print('  Key insight:')
print('    • NLTK / spaCy are ~10-100x faster but miss negation & context')
print('    • DistilBERT catches nuance ("not bad at all" → Positive)')
print('    • Use classical models for low-latency / resource-constrained apps')
print('    • Use Transformers when accuracy is paramount')
print('='*60)
print('  🚀  Run: streamlit run app.py  to launch the dashboard')
print('='*60)